# 1D Ornstein–Uhlenbeck process
##### Kostic et al. 2023 Example 3 Reproduction

$$X_{t+1} = e^{−1} X_{t}+\sqrt{1−e^{−2}}\, \epsilon_t$$

## Dynamical system setup

In [1]:
import numpy as np
import pandas as pd

from kooplearn._linalg import weighted_norm
from kooplearn.preprocessing import FeatureFlattener

x = np.linspace(-4, 4, 1025)[:, None]
flattener = FeatureFlattener()
x_flat = flattener.fit_transform(x)

# This is the exact density used in the paper for OU
density = np.exp(-0.5 * x_flat**2) / np.sqrt(2 * np.pi)


# ------------------------
# generate system dynamics
# ------------------------
def simulate_ou(n_steps, gamma, dt, random_state, x0=0.0):
    """AR(1) sampled from the exact OU transition at physical step size dt."""
    rng = np.random.default_rng(random_state)
    a = np.exp(-gamma * dt)
    b = np.sqrt(1.0 - a**2)
    x = np.empty(n_steps, dtype=float)
    x[0] = float(x0)
    for t in range(n_steps - 1):
        x[t + 1] = a * x[t] + b * rng.standard_normal()
    return pd.DataFrame({"x": x})


# ----------------------------------
# OU specific normalisation constant
# ----------------------------------
def ou_normalisation(functions, x, density):
    #    abs2 = np.abs(functions) ** 2
    #    norms = np.sqrt(np.trapezoid(abs2 * density[:, None], x[:, 0], axis=0))
    W = np.diag(density.reshape(-1))
    norms = weighted_norm(functions, W)
    norms = np.maximum(norms, 1e-12)
    functions = functions / norms
    return functions


# -----------------------------
# OU reference eigenfunctions
# -----------------------------
def compute_ou_eig(gamma, lag, num_components):
    n = np.arange(num_components)
    return np.exp(-n * gamma * lag)


## First, Fig. 2 but with OU

In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
from tqdm import tqdm

from kooplearn.kernel import KernelRidge

# Parameters same as langevin version, where possible
n_train = 2000
subsample = 100
n_steps = n_train * subsample
n_trials = 10
dt = 1e-4
gamma = 1e-4


def fit_and_estimate(reduced_rank, x, density, random_state):
    # Substitute Langevin with OU

    data = simulate_ou(n_steps=n_steps, gamma=gamma, dt=dt, random_state=0, x0=0.0)
    data = data.iloc[::subsample][:n_train]

    # Model definition, same as fig 2 for now
    model = KernelRidge(
        n_components=5,
        reduced_rank=reduced_rank,
        gamma=12.5,
        kernel="rbf",
        alpha=1e-6,
        random_state=random_state,
    )

    # Fit and estimate eigenfunctions
    model.fit(data)  # fit transfer op model
    values, functions = model.eig(
        eval_right_on=x
    )  # (right) eigenvalue estimation, evaluate on array x
    sort_perm = np.flip(np.argsort(np.abs(values)))  # Order decreasingly
    values, functions = values[sort_perm], functions[:, sort_perm]
    functions = ou_normalisation(functions, x, density)
    return functions


# Run functions for both RRR (reduced rank) and kDMD (full rank) estimators

results = defaultdict(list)
for method, reduced_rank in zip(["Principal Components (kDMD)", "Reduced Rank"], [False, True]):
    for i in tqdm(range(n_trials)):
        results[method].append(fit_and_estimate(reduced_rank, x, density, i))

# Print results
fig, axs = plt.subplots(ncols=4, figsize=(9, 2))
for fun_id, ax in enumerate(axs):
    for method, functions in results.items():
        color = "tab:blue" if "Principal" in method else "tab:orange"
        ax.plot(x, functions[0][:, fun_id], color=color, label=method)
    ax.set_title(rf"Eigenfunction $\psi_{fun_id}$", fontsize=9)
    ax.set_xlim(-1, 1)
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncols=2,
    frameon=False,
    bbox_to_anchor=(0.0, 1.05, 1.0, 0.102),
)
plt.tight_layout()
plt.show()


## Figure 1 reproduction
### Kernel family
$$k_{\Pi,\nu}(x,x')=\sum_{i\in\mathbb N}\mu_{\Pi(i)}^{2\nu} f_i(x)f_i(x')$$

#### Building callable Hermite kernel

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from kooplearn.kernel import KernelRidge


# ----------------------------------------------------------------------------------------------
# Hermite helpers / kernel family - in line with Kostic et al. Kernel construction in Example 3
# ----------------------------------------------------------------------------------------------


# -----------------------------
# Hermite features (unchanged)
# -----------------------------
def hermite_features(x, M):
    x = np.asarray(x).reshape(-1)
    f = np.empty((x.shape[0], M))
    f[:, 0] = 1.0
    if M > 1:
        f[:, 1] = x
    for n in range(2, M):
        f[:, n] = (x * f[:, n - 1] - np.sqrt(n - 1) * f[:, n - 2]) / np.sqrt(n)
    return f


# -----------------------------
# FIX 1: bad/ugly share the SAME swap permutation.
# Only nu (the scaling exponent) differs between them.
# -----------------------------
def kernel_permutation(kind, r, M):
    pi = np.arange(M)
    if kind == "good":
        return pi
    if kind in ("bad", "ugly"):
        if 2 * r > M:
            raise ValueError(f"Need M >= 2*r, got M={M}, r={r}")
        pi2 = pi.copy()
        a = np.arange(r)
        b = np.arange(r, 2 * r)
        pi2[a] = b[::-1]
        pi2[b] = a[::-1]
        return pi2
    raise ValueError(kind)


def kernel_weights(kind, r, M):
    mu = np.exp(-np.arange(M))
    if kind == "good":
        nu = 1.0
    elif kind == "bad":
        nu = 1.0 / (r**2)
    elif kind == "ugly":
        nu = float(r**2)
    else:
        raise ValueError(kind)
    pi = kernel_permutation(kind, r, M)
    return mu[pi] ** (2.0 * nu)


def hermite_kernel(kind, r, M):
    w = kernel_weights(kind, r, M)
    cache = {}

    def get_features(x):
        key = np.asarray(x).ravel().tobytes()
        feats = cache.get(key)
        if feats is None:
            feats = hermite_features(x, M)[0]
            cache[key] = feats
        return feats

    def kernel(x, y):
        fx, fy = get_features(x), get_features(y)
        return float(np.sum(w * fx * fy))

    return kernel


In [ ]:
# ---------------------------------------------
# Sign-align estimated functions to a reference
# ~ from kooplearn docs
# ---------------------------------------------


def standardize_sign(eigenfunction, reference, x):
    eigenfunction_cut = cut_functions_to_domain(eigenfunction, x)
    reference_cut = cut_functions_to_domain(reference, x)
    norm_p = np.linalg.norm(eigenfunction_cut + reference_cut)
    norm_m = np.linalg.norm(eigenfunction_cut - reference_cut)
    if norm_p <= norm_m:
        return -1.0 * eigenfunction
    else:
        return eigenfunction


def cut_functions_to_domain(functions, x, x_lims=(-1, 1)):
    mask = (x >= x_lims[0]) & (x <= x_lims[1])
    return functions[mask]


In [ ]:
# -----------------------------
# FIX 2: T=53 truncation, n=20000 samples, gamma_reg=1e-4 (paper values)
# -----------------------------
kernels = ["good", "bad", "ugly"]
methods = [("PCR", False), ("RRR", True)]

n_train = 20000  # was 2000 — paper uses 20,000
subsample = 100
n_steps = n_train * subsample
n_trials = 20
r = 3  # rank / number of leading eigenvalues shown
n_show = 3
M = 53  # was 10 — paper's "good" kernel truncates at T=53
alpha_reg = 1e-4  # was 1e-10 — paper's regularization gamma=1e-4

dt = 1e-4
gamma_ou = 1e-4  # OU decay rate (distinct from regularization)
lag = dt * subsample
true_eigs = compute_ou_eig(gamma_ou, lag, n_show)

x = np.linspace(-4, 4, 1025)[:, None]


def fit_and_estimate_values(reduced_rank, x, random_state, kind, r, M, n_show):
    out = np.full(n_show, np.nan, dtype=float)
    try:
        data = simulate_ou(
            n_steps=n_steps, gamma=gamma_ou, dt=dt, random_state=random_state, x0=0.0
        )
        data = data.iloc[::subsample][:n_train]

        model = KernelRidge(
            n_components=n_show,
            reduced_rank=reduced_rank,
            kernel=hermite_kernel(kind, r, M),
            alpha=alpha_reg,
            random_state=random_state,
        )
        model.fit(data)
        values, _ = model.eig(eval_right_on=x)
        values = np.real_if_close(np.asarray(values))
        if values.ndim == 0:
            values = np.array([values])

        real_mask = np.abs(np.imag(values)) < alpha_reg
        values = np.real(values[real_mask])
        values = values[np.isfinite(values)]
        values = values[values <= 1.0 + alpha_reg]
        values = np.sort(values)[::-1]

        k = min(n_show, values.size)
        out[:k] = values[:k]
    except Exception:
        pass
    return out


results = defaultdict(list)
for kind in kernels:
    for method, reduced_rank in methods:
        for trial in tqdm(range(n_trials), desc=f"{kind}-{method}"):
            vals = fit_and_estimate_values(reduced_rank, x, trial, kind, r, M, n_show)
            results[(kind, method)].append(vals)


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.edgecolor": "0.15",
        "axes.linewidth": 0.8,
        "axes.titlesize": 12,
    }
)

fig, axs = plt.subplots(ncols=3, figsize=(12, 3.2), sharey=False)

# FIX 3: shared log x-axis, e^-5 to e^0, matching paper
log_lo, log_hi = -5, 0
xgrid = np.logspace(log_lo, log_hi, 500)

for col, kind in enumerate(kernels):
    ax = axs[col]

    # FIX 4: pool all estimated eigenvalues per method (no per-mode split)
    for method, color in [
        ("Principal Components (PCR)", "tab:blue"),
        ("Reduced Rank (RRR)", "tab:orange"),
    ]:
        key_method = "PCR" if "PCR" in method else "RRR"
        vals = np.asarray(results[(kind, key_method)], dtype=float).ravel()
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if vals.size < 2:
            continue

        log_vals = np.log(vals)
        kde = gaussian_kde(log_vals, bw_method=0.15)
        log_xgrid = np.log(xgrid)
        density = kde(log_xgrid)

        ax.plot(xgrid, density, color=color, lw=1.6, label=method)
        ax.fill_between(xgrid, density, color=color, alpha=0.15)

    for ev in true_eigs:
        if ev > 0:
            ax.axvline(ev, color="0.4", lw=0.8, zorder=0)

    ax.set_xscale("log")
    ax.set_xlim(10**log_lo, 10**log_hi)
    ax.set_title(f"{kind.capitalize()} kernel", style="italic", fontsize=12)
    ax.set_yticks([])
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig("output/fig1_corrected.png", dpi=200, bbox_inches="tight")
plt.show()


#### Set up, fitting model, collecting spectra

In [ ]:
# Spectral rates paper fig. 1-style eigenfunction comparison:
# - 3 columns = good / bad / ugly kernels
# - overlay PCR (blue) and RRR (orange)
# - show mean estimated eigenfunction across trials for the first few modes

# -----------------------------
# Parameters
# -----------------------------
kernels = ["good", "bad", "ugly"]
methods = [("PCR", False), ("RRR", True)]

n_train = 2000
subsample = 100
n_steps = n_train * subsample
n_trials = 50  # same as paper
n_show = 3  # number of leading eigenfunctions to plot
r = n_show
n_components = 5
M = 10  # kernel truncation level

dt = 1e-4
lag = dt * subsample
gamma = 1e-4  # same as paper
alpha = 1e-10
true_eigs = compute_ou_eig(gamma, lag, n_show)

x_plot = x.reshape(-1)


# -----------------------------
# Fit model and return eigenvalues
# -----------------------------
def fit_and_estimate_values(reduced_rank, x, density, random_state, kind, r, M, n_show):
    out = np.full(n_show, np.nan, dtype=float)

    try:
        data = simulate_ou(
            n_steps=n_steps,
            gamma=gamma,
            dt=dt,
            random_state=random_state,  # IMPORTANT: independent trials
            x0=0.0,
        )
        data = data.iloc[::subsample]
        data = data[:n_train]

        model = KernelRidge(
            n_components=n_show,
            reduced_rank=reduced_rank,
            kernel=hermite_kernel(kind, r, M),
            alpha=alpha,
            random_state=random_state,
        )

        model.fit(data)
        values, functions = model.eig(eval_right_on=x)

        values = np.asarray(values)
        values = np.real_if_close(values)

        if values.ndim == 0:
            values = np.array([values])

        # keep only approximately real eigenvalues
        real_mask = np.abs(np.imag(values)) < alpha
        values = np.real(values[real_mask])

        # keep only finite values
        values = values[np.isfinite(values)]

        # exclude values slightly above 1 from numerical noise
        values = values[values <= 1.0 + alpha]

        # sort by value, largest first
        values = np.sort(values)[::-1]

        k = min(n_show, values.size)
        out[:k] = values[:k]

    except Exception:
        pass

    return out


# -----------------------------
# Collect spectra
# -----------------------------
results = defaultdict(list)

for kind in kernels:
    for method, reduced_rank in methods:
        for trial in tqdm(range(n_trials), desc=f"{kind}-{method}"):
            vals = fit_and_estimate_values(
                reduced_rank=reduced_rank,
                x=x,
                density=density,
                random_state=trial,
                kind=kind,
                r=r,
                M=M,
                n_show=n_show,
            )
            results[(kind, method)].append(np.asarray(vals, dtype=float))


for key, arr_list in results.items():
    arr = np.stack(arr_list, axis=0)
    valid_cols = np.any(np.isfinite(arr), axis=0)
    mean_vals = np.full(arr.shape[1], np.nan)
    mean_vals[valid_cols] = np.nanmean(arr[:, valid_cols], axis=0)
    print(key, mean_vals)


#### Plotting

In [ ]:
# -----------------------------
# Plot
# -----------------------------
from matplotlib.lines import Line2D

true_plot = np.asarray(true_eigs[:n_show], dtype=float)


# Fixed x-range per kernel
kernel_ranges = {
    "good": (0.94, 1.01),
    "bad": (0.92, 1.01),
    "ugly": (-0.02, 1.02),
}


plt.style.use("default")
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.edgecolor": "0.15",
        "axes.linewidth": 0.8,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "font.size": 10,
    }
)


fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(12.6, 5.4),
    sharex=False,
    sharey=False,
)

mode_colors = ["#4C78A8", "#F58518", "#54A24B"]

for row, (method, _) in enumerate(methods):
    for col, kind in enumerate(kernels):
        ax = axs[row, col]
        vals = np.asarray(results[(kind, method)], dtype=float)[:, :n_show]

        xlo, xhi = kernel_ranges[kind]
        bins = np.linspace(xlo, xhi, 70 if kind != "ugly" else 55)

        # distributions of estimated eigenvalues
        for m in range(min(n_show, vals.shape[1])):
            vm = vals[:, m]
            vm = vm[np.isfinite(vm)]
            if vm.size == 0:
                continue

            # light fill
            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="stepfilled",
                color=mode_colors[m],
                alpha=0.16,
                lw=0,
            )
            # clean outline
            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="step",
                color=mode_colors[m],
                lw=1.35,
                label=f"Mode {m + 1}" if (row == 0 and col == 0) else None,
            )

        # true Koopman eigenvalues
        for j, ev in enumerate(true_plot):
            if xlo <= ev <= xhi:
                ax.axvline(
                    float(ev),
                    color="black",
                    lw=0.7,
                    ls=(0, (4, 2)),
                    alpha=1.0,
                    zorder=3,
                    label="Koopman eigenvalue" if (row == 0 and col == 0 and j == 0) else None,
                )

        ax.set_title(f"{method} / {kind}", pad=6)
        ax.set_xlim(xlo, xhi)

        # paper-like axes
        ax.grid(True, axis="y", color="0.88", lw=0.6)
        ax.grid(False, axis="x")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if col == 0:
            ax.set_ylabel("Density")
        if row == len(methods) - 1:
            ax.set_xlabel("Estimated eigenvalue")

# compact legend
legend_handles = [
    Line2D([0], [0], color=mode_colors[0], lw=1.5, label="Mode 1"),
    Line2D([0], [0], color=mode_colors[1], lw=1.5, label="Mode 2"),
    Line2D([0], [0], color=mode_colors[2], lw=1.5, label="Mode 3"),
    Line2D([0], [0], color="black", lw=0.95, ls=(0, (4, 2)), label="Koopman eigenvalue"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    ncol=4,
    frameon=False,
    handlelength=2.2,
    columnspacing=1.6,
)

fig.subplots_adjust(
    top=0.84,
    left=0.07,
    right=0.995,
    bottom=0.12,
    wspace=0.26,
    hspace=0.34,
)

plt.show()
